# 03 · Pump: sensor vs event-camera comparison report

Combines the results of notebooks 01 and 02 into the written comparison `Results/pump_sensor_visual_comparison.md` and the side-by-side confusion-matrix figure. Run 01 and 02 first.

The branches use **different rigs and operating shifts**, so the score gap is not evidence that one modality is better. Laboratory component rigs, not mining machines; sensor and visual branches are evaluated separately (no paired samples, no fusion); the models classify present condition, not remaining useful life or a repair date.

**Script:** `Models/pump_compare_modalities.py`  
**Result file:** `Results/pump_sensor_visual_comparison.md`

In [1]:
REQUIRED = ['Results/pump_bearing_independent.json', 'Results/pump_event_visual.json']
SCRIPT = 'Models/pump_compare_modalities.py'
JSON = 'pump_sensor_visual_comparison.md'
ARGS = []
import contextlib
import io
import json
import os
import runpy
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "Models").is_dir() and (p / "Preprocessing").is_dir())
RAW, PROCESSED, RESULTS = ROOT / "Raw_Data", ROOT / "Processed_Data", ROOT / "Results"

missing = [str(p.relative_to(ROOT)) for p in (ROOT / r for r in REQUIRED) if not p.exists()]
if missing:
    raise FileNotFoundError("Missing local data: " + ", ".join(missing) + ". Run Notebook/00_data_setup.ipynb first.")
for name in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):
    os.environ[name] = "http://127.0.0.1:9"  # offline guard: any network call now fails loudly
os.environ["NO_PROXY"] = ""


def run_script(relative_path, *args, tail=3):
    """Run a project script exactly as `python <script>` would and show its last output lines."""
    path, buffer, saved = ROOT / relative_path, io.StringIO(), sys.argv
    sys.argv = [path.name, *args]
    try:
        with contextlib.redirect_stdout(buffer):
            runpy.run_path(str(path), run_name="__main__")
    finally:
        sys.argv = saved
    text = buffer.getvalue().replace(str(ROOT) + os.sep, "").replace(str(ROOT), ".")
    for line in [l for l in text.splitlines() if l.strip()][-tail:]:
        print("  " + line)


METRICS = ("accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1")


def metric_table(result):
    rows = {}
    def walk(node, path):
        if isinstance(node, dict):
            found = {k: round(node[k], 3) for k in METRICS if isinstance(node.get(k), (int, float))}
            if found:
                rows[" / ".join(path) or "(top level)"] = found
            for key, value in node.items():
                if isinstance(value, dict):
                    walk(value, [*path, key])
    walk(result, [])
    return pd.DataFrame(rows).T


def load(name):
    path = RESULTS / name
    if not path.is_file():
        return None
    text = path.read_text(encoding="utf-8")
    return json.loads(text) if name.endswith(".json") else text


print(f"Repository root: {ROOT.name}/ | data present | offline guard on")

Repository root: capstone-week4/ | data present | offline guard on


In [2]:
committed = load(JSON)
run_script(SCRIPT, *ARGS)
result = load(JSON)
if JSON.endswith(".json"):
    same = committed is not None and metric_table(committed).equals(metric_table(result))
else:
    same = committed == result
print(f"\n{JSON}: {'matches the committed result' if same else 'differs from the committed result (see note above)'}")
if JSON.endswith(".json"):
    display(metric_table(result))

  Saved Results\pump_sensor_visual_comparison.md and Figures\pump_sensor_visual_confusions.png

pump_sensor_visual_comparison.md: matches the committed result


![Result figure](../Figures/pump_sensor_visual_confusions.png)

*The figure shown is the committed version in `Figures/`; running this notebook regenerates it.*

Full write-up: [`Results/pump_sensor_visual_comparison.md`](../Results/pump_sensor_visual_comparison.md).